In [0]:
from pyspark.sql import functions as F

# Read Silver Fact Work and dimensions
work = spark.table(
    "workforce_analytics.silver.fact_work"
).alias("f")

people = spark.table(
    "workforce_analytics.silver.dim_people"
).alias("p")

county = spark.table(
    "workforce_analytics.silver.dim_county"
).alias("c")

cancellation = spark.table(
    "workforce_analytics.silver.dim_cancellation_reason"
).alias("r")

# Join fact to dimensions
fact_work_gold = (
    work
    .join(
        people,
        F.col("f.people_id") == F.col("p.people_id"),
        "left"
    )
    .join(
        county,
        F.col("f.county_id") == F.col("c.county_id"),
        "left"
    )
    .join(
        cancellation,
        F.col("f.cancellation_id") == F.col("r.cancellation_id"),
        "left"
    )
    .select(
        # Original fact columns
        *[F.col(f"f.{col}") for col in work.columns],

        # People attributes
        F.col("p.employee_name"),
        F.col("p.job_role"),
        F.col("p.employment_type"),
        F.col("p.team"),

        # County attributes
        F.col("c.county_name"),
        F.col("c.region"),
        F.col("c.country"),
        F.col("c.population_band"),
        F.col("c.urban_rural"),

        # Cancellation attributes
        F.col("r.cancellation_reason"),
        F.col("r.reason_group"),
        F.col("r.is_customer_driven"),
        F.col("r.severity")
    )
)

# Save Gold table
(
    fact_work_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workforce_analytics.gold.fact_work_gold")
)

print("Gold Fact Work created successfully.")

In [0]:
gold_work = spark.table(
    "workforce_analytics.gold.fact_work_gold"
)

print("Gold row counts by snapshot date:")

display(
    gold_work.groupBy("snapshot_date")
    .count()
    .orderBy("snapshot_date")
)

print("Total rows:", gold_work.count())

print("Duplicate work IDs within snapshots:")

display(
    gold_work.groupBy("snapshot_date", "work_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Missing employee names:", 
      gold_work.filter(F.col("employee_name").isNull()).count())

print("Missing county names:",
      gold_work.filter(F.col("county_name").isNull()).count())